# ConvNeXt-Only Robustness (FashionStyle14)

## Objective
Fine-tune **ConvNeXt-Base** (`convnext_base` in timm) on **FashionStyle14** across **30 stratified random splits** (seeds from `seeds_list.txt`), with the same MLP head and training protocol as other image-only baselines.

## Pipeline
1. Load **30 seeds**, hyperparameters, and paths.
2. For **each seed**: 70% / 15% / 15% stratified split, train with early stopping, evaluate on test.
3. **Print and save** per-seed learning curves (`learning_curves.png`).
4. Aggregate **overall** test metrics (mean +/- std) and **per-style** F1 (mean +/- std).

## Google Colab
- Mount Google Drive, then set runtime to **GPU**.
- **Data**: `/content/drive/Shareddrives/DATA255/FashionStyle14_v1`
- **Results**: `./results/imageonly_convnext/` (under `/content` in Colab)
- Runs all **30** seeds from `seeds_list.txt`.

## Inputs
- `FashionStyle14_v1/complete_dataset.csv` (on Shared Drive)
- `FashionStyle14_v1/seeds_list.txt`
- `FashionStyle14_v1/dataset/`

## Outputs
**`results/imageonly_convnext/`**

| Per seed (`seed_<seed>/`) | Aggregated |
|---------------------------|------------|
| `best_model.pt` | `all_seeds_summary.csv` |
| `training_history.json` | `aggregation_summary.csv` |
| `learning_curves.png` | `per_class_metrics_summary.csv` |
| `test_metrics.json`, `per_class.csv` | |


## 0. Google Colab: mount Drive


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q timm


In [11]:
import shutil
from pathlib import Path

drive_data_root = Path("/content/drive/Shareddrives/My_DATA255/FashionStyle14_v1")
local_data_root = Path("/content/FashionStyle14_v1")

if not local_data_root.exists():
    shutil.copytree(drive_data_root, local_data_root)

DATA_ROOT = local_data_root

## 1. Configuration, imports, and paths


In [12]:
from __future__ import annotations

import json
import os
import random
import re
import warnings
from pathlib import Path
from typing import Any, Dict, List, Tuple
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore", category=UserWarning)

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

LEARNING_RATE = 5e-5
BATCH_SIZE = 32
MAX_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 5
DROPOUT = 0.5
WEIGHT_DECAY = 1e-4
MODEL_INIT_SEED = 42
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15

MODEL_LABEL = "convnext"
CONVNEXT_MODEL_NAME = "convnext_base"

random.seed(MODEL_INIT_SEED)
np.random.seed(MODEL_INIT_SEED)
torch.manual_seed(MODEL_INIT_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(MODEL_INIT_SEED)


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "notebooks" / "robustness").is_dir() and (candidate / "README.md").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not locate fashion-multimodal-fusion repo root. "
        "Open the notebook from the repo root or notebooks/robustness/."
    )


def find_data_dir(repo_root: Path) -> Path:
    candidates = [
        repo_root.parent / "FusionStyle" / "FashionStyle14_v1",
        repo_root / "FusionStyle" / "FashionStyle14_v1",
        Path.cwd() / "FusionStyle" / "FashionStyle14_v1",
        Path.cwd().parent / "FusionStyle" / "FashionStyle14_v1",
    ]
    for data_dir in candidates:
        dataset_dir = data_dir / "dataset"
        if data_dir.is_dir() and dataset_dir.is_dir() and (data_dir / "complete_dataset.csv").is_file():
            return data_dir.resolve()
    raise FileNotFoundError(
        "Could not locate FusionStyle/FashionStyle14_v1 with complete_dataset.csv and dataset/."
    )


def load_seeds(seeds_file: Path) -> List[int]:
    if not seeds_file.is_file():
        raise FileNotFoundError(f"Missing seeds file: {seeds_file}")
    content = seeds_file.read_text(encoding="utf-8")
    matches = re.findall(r"Seed\s+(\d+)", content, flags=re.IGNORECASE)
    seeds = sorted({int(s) for s in matches if 1 <= int(s) <= 500})
    if len(seeds) != 30:
        print(f"Warning: expected 30 seeds, found {len(seeds)}")
    return seeds[:30]


if IN_COLAB:
    DRIVE_DATA_DIR = Path("/content/drive/Shareddrives/My_DATA255/FashionStyle14_v1")
    LOCAL_DATA_DIR = Path("/content/FashionStyle14_v1")

    if not LOCAL_DATA_DIR.exists():
        print("Copying dataset from Google Drive to local Colab disk...")
        shutil.copytree(DRIVE_DATA_DIR, LOCAL_DATA_DIR)
        print("Dataset copy completed.")
    else:
        print("Local dataset already exists. Using local copy.")

    DATA_DIR = LOCAL_DATA_DIR
    RESULTS_ROOT = Path("/content/drive/Shareddrives/My_DATA255/results/imageonly_convnext").resolve()

else:
    REPO_ROOT = find_repo_root()
    DATA_DIR = find_data_dir(REPO_ROOT)
    RESULTS_ROOT = (REPO_ROOT / "results" / "imageonly_convnext").resolve()

IMAGE_ROOT = DATA_DIR
SEEDS_FILE = DATA_DIR / "seeds_list.txt"
COMPLETE_CSV = DATA_DIR / "complete_dataset.csv"
SEEDS = load_seeds(SEEDS_FILE)

if not IN_COLAB:
    print("REPO_ROOT:", REPO_ROOT)
print("IN_COLAB:", IN_COLAB)
print("DATA_DIR:", DATA_DIR)
print("RESULTS_ROOT:", RESULTS_ROOT)
print("IMAGE_ROOT / dataset exists:", (IMAGE_ROOT / "dataset").is_dir())
print(f"Loaded {len(SEEDS)} seeds:", SEEDS)

Local dataset already exists. Using local copy.
IN_COLAB: True
DATA_DIR: /content/FashionStyle14_v1
RESULTS_ROOT: /content/drive/Shareddrives/My_DATA255/results/imageonly_convnext
IMAGE_ROOT / dataset exists: True
Loaded 30 seeds: [13, 14, 16, 17, 45, 48, 53, 58, 72, 102, 112, 115, 120, 126, 141, 215, 217, 259, 280, 288, 303, 309, 328, 333, 347, 360, 367, 378, 380, 457]


## 2. Hardware check and runtime settings


In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"

if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    total_mem_gb = props.total_memory / (1024**3)
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"Total GPU memory: {total_mem_gb:.2f} GB")
    if total_mem_gb < 8:
        BATCH_SIZE = 8
    elif total_mem_gb < 12:
        BATCH_SIZE = 16
    else:
        BATCH_SIZE = 32
else:
    print("CUDA not available; using CPU (training will be slow).")
    BATCH_SIZE = 8

print(f"Selected BATCH_SIZE: {BATCH_SIZE}")
print(f"Using AMP on CUDA: {use_amp}")


CUDA device: NVIDIA A100-SXM4-40GB
Total GPU memory: 39.49 GB
Selected BATCH_SIZE: 32
Using AMP on CUDA: True


## 3. Load full dataset, labels, and stratified split helper


In [14]:
def normalize_rel_path(path_str: str) -> str:
    return str(path_str).strip().replace("\\", "/")

def load_complete_dataset(csv_path: Path) -> pd.DataFrame:
    lines = csv_path.read_text(encoding="utf-8").splitlines()
    rel = [ln.strip() for ln in lines if ln.strip()]
    df = pd.DataFrame({"rel_path": rel})
    df["rel_path"] = df["rel_path"].map(normalize_rel_path)
    df["style"] = df["rel_path"].str.split("/").str[1]
    df["abs_path"] = df["rel_path"].apply(lambda r: str((IMAGE_ROOT / r.replace("/", os.sep)).resolve()))
    df = df[df["abs_path"].map(os.path.isfile)].reset_index(drop=True)
    return df


df_full = load_complete_dataset(COMPLETE_CSV)
classes = sorted(df_full["style"].unique().tolist())
assert len(classes) == 14, f"Expected 14 classes, got {len(classes)}: {classes}"
style_to_idx = {s: i for i, s in enumerate(classes)}
num_classes = len(classes)

print("Full dataset size (existing files only):", len(df_full))
print("Number of classes:", num_classes)
print("Classes:", classes)
print(f"Data splits will be created per seed ({len(SEEDS)} experiments).")


def split_by_seed(df: pd.DataFrame, seed_value: int) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train_df, temp_df = train_test_split(
        df,
        test_size=(VAL_RATIO + TEST_RATIO),
        stratify=df["style"],
        random_state=seed_value,
    )
    val_df, test_df = train_test_split(
        temp_df,
        test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
        stratify=temp_df["style"],
        random_state=seed_value,
    )
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)

Full dataset size (existing files only): 13212
Number of classes: 14
Classes: ['conservative', 'dressy', 'ethnic', 'fairy', 'feminine', 'gal', 'girlish', 'kireime-casual', 'lolita', 'mode', 'natural', 'retro', 'rock', 'street']
Data splits will be created per seed (30 experiments).


## 4. PyTorch dataset and ConvNeXt transforms


In [15]:
class FashionImageDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, style_to_idx: Dict[str, int], transform):
        self.frame = frame.reset_index(drop=True)
        self.style_to_idx = style_to_idx
        self.transform = transform

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.frame.iloc[idx]
        path = row["abs_path"]
        style = row["style"]
        label = self.style_to_idx[style]
        try:
            img = Image.open(path).convert("RGB")
        except Exception:
            img = Image.new("RGB", (224, 224), color=(0, 0, 0))
        tensor = self.transform(img)
        return {"pixel_values": tensor, "label": label}


def build_convnext_transform():
    data_config = timm.data.resolve_data_config({}, model=CONVNEXT_MODEL_NAME, verbose=False)
    return timm.data.create_transform(**data_config, is_training=False)


convnext_transform = build_convnext_transform()
NUM_WORKERS = 2

def make_loaders(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    transform,
) -> Tuple[DataLoader, DataLoader, DataLoader]:
    train_ds = FashionImageDataset(train_df, style_to_idx, transform)
    val_ds = FashionImageDataset(val_df, style_to_idx, transform)
    test_ds = FashionImageDataset(test_df, style_to_idx, transform)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    return train_loader, val_loader, test_loader


print("ConvNeXt transform ready. Batch size:", BATCH_SIZE)


ConvNeXt transform ready. Batch size: 32


## 5. ConvNeXt model


In [16]:
class ConvNeXtImageClassifier(nn.Module):
    """timm ConvNeXt backbone + MLP head (same pattern as other image-only models)."""

    def __init__(self, num_classes: int, dropout: float = DROPOUT):
        super().__init__()
        self.backbone = timm.create_model(CONVNEXT_MODEL_NAME, pretrained=True, num_classes=0)
        in_dim = self.backbone.num_features
        self.head = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.backbone(x))


def build_convnext_model() -> ConvNeXtImageClassifier:
    return ConvNeXtImageClassifier(num_classes=num_classes)


print("ConvNeXtImageClassifier defined.")


ConvNeXtImageClassifier defined.


## 6. Training, evaluation, and learning curves (saved)


In [17]:
def evaluate(model: nn.Module, loader: DataLoader, criterion, device: torch.device, use_amp: bool) -> Tuple[float, Dict[str, float], np.ndarray, np.ndarray]:
    model.eval()
    total_loss = 0.0
    all_preds: List[int] = []
    all_labels: List[int] = []
    with torch.no_grad():
        for batch in loader:
            x = batch["pixel_values"].to(device, non_blocking=True)
            y = batch["label"].to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, enabled=use_amp and device.type == "cuda"):
                logits = model(x)
                loss = criterion(logits, y)
            total_loss += float(loss.item()) * y.size(0)
            preds = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(y.cpu().numpy().tolist())
    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    macro_p = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    macro_r = recall_score(all_labels, all_preds, average="macro", zero_division=0)
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    metrics = {"loss": avg_loss, "accuracy": acc, "macro_precision": macro_p, "macro_recall": macro_r, "macro_f1": macro_f1}
    return avg_loss, metrics, np.array(all_preds), np.array(all_labels)


def train_one_epoch(model, loader, optimizer, criterion, device, use_amp) -> Tuple[float, float]:
    model.train()
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp and device.type == "cuda")
    total_loss = 0.0
    correct = 0
    total = 0
    for batch in loader:
        x = batch["pixel_values"].to(device, non_blocking=True)
        y = batch["label"].to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=use_amp and device.type == "cuda"):
            logits = model(x)
            loss = criterion(logits, y)
        if scaler.is_enabled():
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()
        total_loss += float(loss.item()) * y.size(0)
        preds = logits.argmax(dim=1)
        correct += int((preds == y).sum().item())
        total += int(y.size(0))
    return total_loss / max(total, 1), correct / max(total, 1)


def plot_learning_curves(history: Dict[str, Any], title: str, save_path: Path | None = None) -> None:
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(epochs, history["train_loss"], label="train")
    axes[0].plot(epochs, history["val_loss"], label="val")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("epoch")
    axes[0].legend()
    axes[1].plot(epochs, history["val_macro_f1"], label="val macro F1")
    axes[1].plot(epochs, history["train_acc"], label="train acc", alpha=0.7)
    axes[1].plot(epochs, history["val_acc"], label="val acc", alpha=0.7)
    axes[1].set_title("Metrics")
    axes[1].set_xlabel("epoch")
    axes[1].legend()
    fig.suptitle(title)
    fig.tight_layout()
    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved learning curves: {save_path}")
    plt.close(fig)


def set_model_init_seed() -> None:
    random.seed(MODEL_INIT_SEED)
    np.random.seed(MODEL_INIT_SEED)
    torch.manual_seed(MODEL_INIT_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(MODEL_INIT_SEED)


def train_image_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    test_loader: DataLoader,
    out_dir: Path,
    seed_value: int,
    device: torch.device,
    use_amp: bool,
) -> Dict[str, Any]:
    out_dir.mkdir(parents=True, exist_ok=True)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    history = {"train_loss": [], "val_loss": [], "val_macro_f1": [], "train_acc": [], "val_acc": []}
    best_f1 = -1.0
    best_state = None
    patience_left = EARLY_STOPPING_PATIENCE

    for epoch in range(1, MAX_EPOCHS + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, device, use_amp)
        val_loss, val_metrics, _, _ = evaluate(model, val_loader, criterion, device, use_amp)
        history["train_loss"].append(tr_loss)
        history["val_loss"].append(val_loss)
        history["val_macro_f1"].append(val_metrics["macro_f1"])
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(val_metrics["accuracy"])
        print(
            f"  Epoch {epoch:02d} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
            f"val loss {val_loss:.4f} acc {val_metrics['accuracy']:.4f} macroF1 {val_metrics['macro_f1']:.4f}"
        )
        if val_metrics["macro_f1"] > best_f1:
            best_f1 = val_metrics["macro_f1"]
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
            patience_left = EARLY_STOPPING_PATIENCE
        else:
            patience_left -= 1
            if patience_left <= 0:
                print("  Early stopping triggered.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    torch.save(model.state_dict(), out_dir / "best_model.pt")
    with open(out_dir / "training_history.json", "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2)

    curve_path = out_dir / "learning_curves.png"
    plot_learning_curves(history, title=f"ConvNeXt (seed {seed_value})", save_path=curve_path)

    _, test_metrics, test_preds, test_labels = evaluate(model, test_loader, criterion, device, use_amp)
    per_class_precision = precision_score(test_labels, test_preds, average=None, zero_division=0)
    per_class_recall = recall_score(test_labels, test_preds, average=None, zero_division=0)
    per_class_f1 = f1_score(test_labels, test_preds, average=None, zero_division=0)

    per_class_rows = []
    for i, cls in enumerate(classes):
        per_class_rows.append({
            "class": cls,
            "acc": float(per_class_recall[i]),
            "precision": float(per_class_precision[i]),
            "recall": float(per_class_recall[i]),
            "f1": float(per_class_f1[i]),
        })
    per_class_df = pd.DataFrame(per_class_rows, columns=["class", "acc", "precision", "recall", "f1"])

    with open(out_dir / "test_metrics.json", "w", encoding="utf-8") as f:
        json.dump({k: float(v) for k, v in test_metrics.items()}, f, indent=2)
    per_class_csv_path = out_dir / "per_class.csv"
    per_class_df.to_csv(per_class_csv_path, index=False, encoding="utf-8")

    print(f"\n[ConvNeXt | seed {seed_value}] Test metrics:", test_metrics)
    print(f"[ConvNeXt | seed {seed_value}] Per-class metrics (saved to {per_class_csv_path}):")
    print(per_class_df.to_string(index=False))

    return {"seed": seed_value, "test_metrics": test_metrics, "per_class_df": per_class_df}


def run_convnext_robustness() -> pd.DataFrame:
    RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
    rows: List[Dict[str, Any]] = []

    for seed_idx, seed_value in enumerate(SEEDS, start=1):
        seed_dir = RESULTS_ROOT / f"seed_{seed_value}"
        done_marker = seed_dir / "test_metrics.json"
        if done_marker.is_file():
            print(f"[ConvNeXt] Seed {seed_value} ({seed_idx}/{len(SEEDS)}): skip (already done)")
            with open(done_marker, encoding="utf-8") as f:
                cached = json.load(f)
            rows.append({"seed": seed_value, **cached})
            continue

        print("=" * 70)
        print(f"[ConvNeXt] Experiment {seed_idx}/{len(SEEDS)} | data split seed = {seed_value}")
        print("=" * 70)
        train_df, val_df, test_df = split_by_seed(df_full, seed_value)
        print(f"  Split sizes: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")
        train_loader, val_loader, test_loader = make_loaders(train_df, val_df, test_df, convnext_transform)

        set_model_init_seed()
        model = build_convnext_model().to(device)
        result = train_image_model(model, train_loader, val_loader, test_loader, seed_dir, seed_value, device, use_amp)
        rows.append({"seed": seed_value, **{k: float(v) for k, v in result["test_metrics"].items()}})

        del model, train_loader, val_loader, test_loader
        if device.type == "cuda":
            torch.cuda.empty_cache()

    summary_df = pd.DataFrame(rows)
    summary_path = RESULTS_ROOT / "all_seeds_summary.csv"
    summary_df.to_csv(summary_path, index=False, encoding="utf-8")
    print(f"\n[ConvNeXt] Saved per-seed summary: {summary_path}")
    return summary_df


def print_and_save_aggregates(summary_df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    metric_cols = ["accuracy", "macro_precision", "macro_recall", "macro_f1"]
    agg_rows = []
    print("\n" + "=" * 70)
    print("OVERALL TEST METRICS (mean +/- std across seeds)")
    print("=" * 70)
    for col in metric_cols:
        if col not in summary_df.columns:
            continue
        mean_v = float(summary_df[col].mean())
        std_v = float(summary_df[col].std())
        agg_rows.append({"metric": col, "mean": mean_v, "std": std_v})
        print(f"  {col}: {mean_v:.4f} +/- {std_v:.4f}")
    agg_df = pd.DataFrame(agg_rows)
    agg_path = RESULTS_ROOT / "aggregation_summary.csv"
    agg_df.to_csv(agg_path, index=False, encoding="utf-8")
    print(f"\nSaved: {agg_path}")

    per_class_rows = []
    for seed_value in SEEDS:
        pc_path = RESULTS_ROOT / f"seed_{seed_value}" / "per_class.csv"
        if not pc_path.is_file():
            continue
        pc = pd.read_csv(pc_path)
        for _, row in pc.iterrows():
            per_class_rows.append({
                "seed": seed_value,
                "style": row["class"],
                "f1": row["f1"],
                "precision": row["precision"],
                "recall": row["recall"],
                "acc": row["acc"],
            })
    if not per_class_rows:
        return agg_df, pd.DataFrame()

    pc_long = pd.DataFrame(per_class_rows)
    style_summary = []
    print("\n" + "=" * 70)
    print("PER-STYLE TEST F1 (mean +/- std across seeds)")
    print("=" * 70)
    for style, grp in pc_long.groupby("style"):
        f1_mean = float(grp["f1"].mean())
        f1_std = float(grp["f1"].std())
        style_summary.append({
            "style": style,
            "f1_mean": f1_mean,
            "f1_std": f1_std,
            "precision_mean": float(grp["precision"].mean()),
            "recall_mean": float(grp["recall"].mean()),
            "acc_mean": float(grp["acc"].mean()),
        })
        print(f"  {style:20s}  F1 {f1_mean:.4f} +/- {f1_std:.4f}")

    style_df = pd.DataFrame(style_summary).sort_values("style").reset_index(drop=True)
    style_path = RESULTS_ROOT / "per_class_metrics_summary.csv"
    style_df.to_csv(style_path, index=False, encoding="utf-8")
    print(f"\nSaved: {style_path}")
    return agg_df, style_df


print("Training utilities ready.")

Training utilities ready.


## 7. Run ConvNeXt robustness: 30 seeds


In [18]:
summary_convnext = run_convnext_robustness()

[ConvNeXt] Experiment 1/30 | data split seed = 13
  Split sizes: train=9248, val=1982, test=1982
  Epoch 01 | train loss 1.9209 acc 0.3861 | val loss 1.0623 acc 0.6993 macroF1 0.6934
  Epoch 02 | train loss 0.9540 acc 0.7096 | val loss 0.7156 acc 0.7689 macroF1 0.7682
  Epoch 03 | train loss 0.4742 acc 0.8701 | val loss 0.6721 acc 0.7891 macroF1 0.7937
  Epoch 04 | train loss 0.2542 acc 0.9369 | val loss 0.6908 acc 0.8103 macroF1 0.8113
  Epoch 05 | train loss 0.1325 acc 0.9704 | val loss 0.8296 acc 0.7936 macroF1 0.7964
  Epoch 06 | train loss 0.0831 acc 0.9835 | val loss 0.9922 acc 0.7962 macroF1 0.7972
  Epoch 07 | train loss 0.0532 acc 0.9892 | val loss 1.1128 acc 0.7926 macroF1 0.7919
  Epoch 08 | train loss 0.0547 acc 0.9879 | val loss 1.0405 acc 0.7997 macroF1 0.8012
  Epoch 09 | train loss 0.0455 acc 0.9903 | val loss 1.0588 acc 0.8042 macroF1 0.8065
  Early stopping triggered.
  Saved learning curves: /content/drive/Shareddrives/My_DATA255/results/imageonly_convnext/seed_13/le

In [19]:
agg_df, per_style_df = print_and_save_aggregates(summary_convnext)
display(agg_df)
display(per_style_df.head(14))


OVERALL TEST METRICS (mean +/- std across seeds)
  accuracy: 0.8044 +/- 0.0102
  macro_precision: 0.8141 +/- 0.0106
  macro_recall: 0.8046 +/- 0.0100
  macro_f1: 0.8067 +/- 0.0101

Saved: /content/drive/Shareddrives/My_DATA255/results/imageonly_convnext/aggregation_summary.csv

PER-STYLE TEST F1 (mean +/- std across seeds)
  conservative          F1 0.7306 +/- 0.0238
  dressy                F1 0.9498 +/- 0.0135
  ethnic                F1 0.8606 +/- 0.0242
  fairy                 F1 0.9357 +/- 0.0178
  feminine              F1 0.7627 +/- 0.0330
  gal                   F1 0.8019 +/- 0.0270
  girlish               F1 0.6437 +/- 0.0295
  kireime-casual        F1 0.6770 +/- 0.0297
  lolita                F1 0.9539 +/- 0.0157
  mode                  F1 0.7791 +/- 0.0285
  natural               F1 0.8052 +/- 0.0310
  retro                 F1 0.7547 +/- 0.0311
  rock                  F1 0.7927 +/- 0.0341
  street                F1 0.8458 +/- 0.0296

Saved: /content/drive/Shareddrives/My_DATA2

,metric,mean,std
0,accuracy,0.804440,0.010194
1,macro_precision,0.814064,0.010564
2,macro_recall,0.804632,0.009969
3,macro_f1,0.806669,0.010130


,style,f1_mean,f1_std,precision_mean,recall_mean,acc_mean
0,conservative,0.730643,0.023827,0.734560,0.733002,0.733002
1,dressy,0.949818,0.013542,0.969872,0.931817,0.931817
2,ethnic,0.860580,0.024215,0.883887,0.842119,0.842119
3,fairy,0.935659,0.017813,0.927156,0.945851,0.945851
4,feminine,0.762703,0.033036,0.786004,0.748760,0.748760
5,gal,0.801886,0.027009,0.826907,0.782751,0.782751
6,girlish,0.643664,0.029515,0.626360,0.669478,0.669478
7,kireime-casual,0.676964,0.029689,0.663160,0.698101,0.698101
8,lolita,0.953867,0.015661,0.963026,0.945926,0.945926
9,mode,0.779094,0.028509,0.767624,0.796855,0.796855
